In [1]:
!source venv/bin/activate


In [2]:
import json 

import pandas as pd
import re

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/var/folders/c7/m5m1t1ld37s0__305c38wmr00000gn/T/ipykernel_19045/1393976005.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
W1122 21:21:53.614000 19045 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using imag

View ID and class label

In [ ]:


with open("docs/webvowl/data/ontology_full.json") as f:
# with open("docs/webvowl/data/ontology.json") as f:
    data = json.load(f)

def get_name(attr):
    label = attr.get("label")
    if isinstance(label, str) and label.strip():
        return label.strip()
    if isinstance(label, dict):
        for key in ("en", "undefined", "IRI-based"):
            val = label.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        for val in label.values():
            if isinstance(val, str) and val.strip():
                return val.strip()
    iri = attr.get("iri", "")
    return iri.split("#")[-1] if "#" in iri else iri.split("/")[-1]

id_to_name = {
    attr["id"]: get_name(attr)
    for attr in data.get("classAttribute", [])
    if "id" in attr
}
print(len(id_to_name))

sorted_id_to_name = dict(sorted(id_to_name.items(), key=lambda item: int(item[0])))
# print(sorted_id_to_name)

for cid, name in list(sorted_id_to_name.items())[:25]:
    print(f"cid: {cid:>4} -> {name}")

244
cid:    1 -> BMI category
cid:    2 -> Body height observation
cid:    4 -> Medical condition
cid:    5 -> Model
cid:    6 -> dateTimeStamp
cid:    8 -> Atrial fibrillation risk status
cid:    9 -> Atrial fibrillation
cid:   11 -> Anthropometric property
cid:   12 -> Biometric property
cid:   13 -> duration
cid:   15 -> Atrial fibrillation health risk assessment
cid:   17 -> ECG observation
cid:   18 -> ECG sensor
cid:   20 -> string
cid:   22 -> Health risk relationship
cid:   23 -> Unit of measure
cid:   25 -> ECG factor
cid:   28 -> decimal
cid:   30 -> Wearable device
cid:   32 -> integer
cid:   34 -> Observation
cid:   36 -> Evidence
cid:   38 -> Publication
cid:   40 -> Artificial neural network
cid:   41 -> Literal


In [4]:


with open("docs/webvowl/data/ontology_full.json") as f:
# with open("docs/webvowl/data/ontology.json") as f:
    data = json.load(f)
print("Top keys:", list(data.keys()))
print("Num classes:", len(data.get("class", [])), "Num classAttrs:", len(data.get("classAttribute", [])))
print("Sample classAttrs:", [(c.get("id"), c.get("label")) for c in data.get("classAttribute", [])[:10]])


Top keys: ['_comment', 'header', 'namespace', 'class', 'classAttribute', 'property', 'propertyAttribute']
Num classes: 244 Num classAttrs: 244
Sample classAttrs: [('6', {'IRI-based': 'dateTimeStamp'}), ('13', {'IRI-based': 'duration'}), ('20', {'IRI-based': 'string'}), ('28', {'IRI-based': 'decimal'}), ('32', {'IRI-based': 'integer'}), ('42', {'IRI-based': 'float'}), ('43', {'IRI-based': 'dateTimeStamp'}), ('44', {'IRI-based': 'float'}), ('45', {'IRI-based': 'int'}), ('48', {'IRI-based': 'int'})]


In [ ]:
#this shows that the labels can be under en or undefined or iri-based labels
with open("docs/webvowl/data/ontology_full.json") as f:
    data = json.load(f)

keywords = ("age", "demo", "sex", "race", "fibrillation", "afmo")

for attr in data.get("classAttribute", []):
    blob = json.dumps(attr).lower()
    if any(k in blob for k in keywords):
        print(f"id={attr.get('id'):>4} | label={attr.get('label')} | iri={attr.get('iri')}")

id=  87 | label={'IRI-based': 'ConferenceProceedings', 'en': 'Conference proceedings'} | iri=http://purl.org/spar/fabio/ConferenceProceedings
id= 108 | label={'IRI-based': 'JournalVolume', 'en': 'Journal volume'} | iri=http://purl.org/spar/fabio/JournalVolume
id=  11 | label={'IRI-based': 'AnthropometricProperty', 'undefined': 'Anthropometric property'} | iri=http://purl.org/phmo#AnthropometricProperty
id= 123 | label={'IRI-based': 'BayesianNetwork', 'undefined': 'Bayesian network'} | iri=http://purl.org/phmo#BayesianNetwork
id= 127 | label={'IRI-based': 'WorkshopProceedings', 'en': 'Workshop proceedings'} | iri=http://purl.org/spar/fabio/WorkshopProceedings
id= 128 | label={'IRI-based': 'MAXO_0000092', 'en': 'Excessive alcohol avoidance'} | iri=http://purl.obolibrary.org/obo/MAXO_0000092
id= 162 | label={'IRI-based': 'Clinician', 'undefined': 'Clinician'} | iri=http://purl.org/phmo#Clinician
id= 101 | label={'IRI-based': 'MonitoredPerson', 'undefined': 'Monitored person'} | iri=http:/

Intersectin Pipeline Plan:
1. start with word matching
2. run a text embedding model to get similarity scores
3. run a LLM to find intersecting class labels

In [ ]:
#exact word match

csv_path = "AFib_Clinical_Metadata_for_Jet.xlsx - Data Dictionary.csv"

df = pd.read_csv(csv_path, index_col=0)

columns = df["column"].dropna().astype(str).tolist()

def normalize_text(text: str) -> str:
    """Lowercase, strip punctuation/extra spaces, and normalize common spelling variants."""
    text = text.lower().strip()
    # text = text.replace("apnoea", "apnea")
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# map normalized column names to original column names
norm_to_cols: dict[str, list[str]] = {}
for col in columns:
    norm_col = normalize_text(col)
    if norm_col:
        norm_to_cols.setdefault(norm_col, []).append(col)

# print("norm_to_cols: ", norm_to_cols)

exact_matches = []

for cid, label in sorted_id_to_name.items():
    norm_label = normalize_text(str(label))
    # print("n lab: ", norm_label)
    if norm_label in norm_to_cols:
       
        for matched_col in norm_to_cols[norm_label]:
            exact_matches.append(
                {
                    "afmo_id": cid,
                    "afmo_label": label,
                    "norm_label": norm_label,
                    "metadata_column": matched_col,
                }
            )

matches_df = pd.DataFrame(exact_matches)
print(f"Found {len(matches_df)} exact normalized matches:")
print(matches_df)


matched_afmo_labels = matches_df["afmo_label"].unique().tolist() if not matches_df.empty else []


Found 1 exact normalized matches:
  afmo_id    afmo_label    norm_label metadata_column
0     138  Hypertension  hypertension    Hypertension


In [ ]:


# Build text descriptions for each clinical metadata column
def build_col_text(row: pd.Series) -> str:
    parts = [
        f"Column: {row.get('column', '')}",
        f"Meaning: {row.get('meaning', '')}",
        f"Coding/Units: {row.get('units_or_coding', '')}",
        f"Target vocabulary: {row.get('vocabulary_target', '')}",
    ]
    return " | ".join(str(p).strip() for p in parts if pd.notna(p) and str(p).strip())

meta_rows = df.dropna(subset=["column"]).copy()
meta_rows["embed_text"] = meta_rows.apply(build_col_text, axis=1)

# Build text representations for each AFMO class
afmo_items = [(str(cid), str(label)) for cid, label in sorted_id_to_name.items()]
afmo_ids = [cid for cid, _ in afmo_items]
afmo_labels = [label for _, label in afmo_items]
afmo_texts = [f"AFMO ontology class: {label}" for label in afmo_labels]


# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("neuml/pubmedbert-base-embeddings")

col_embs = model.encode(
    meta_rows["embed_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)
afmo_embs = model.encode(
    afmo_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# embeddings are L2-normalized sodot product == cosine similarity
sim_matrix = np.matmul(col_embs, afmo_embs.T)  # shape: (n_cols, n_afmo_classes)

# top-K AFMO matches per clinical metadata column
TOP_K = 3
results = []

for i, (_, mrow) in enumerate(meta_rows.iterrows()):
    scores = sim_matrix[i]
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, j in enumerate(top_indices, start=1):
        results.append(
            {
                "metadata_col": mrow["column"],
                "rank": rank,
                "afmo_id": afmo_ids[j],
                "afmo_label": afmo_labels[j],
                "cosine_similarity": round(float(scores[j]), 4),
            }
        )

sim_df = pd.DataFrame(results)
print(sim_df.head(15).to_string(index=False))



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

     metadata_col  rank afmo_id                             afmo_label  cosine_similarity
        Sample ID     1      76                                decimal             0.4512
        Sample ID     2     530                                decimal             0.4512
        Sample ID     3      28                                decimal             0.4512
 Date of ablation     1       9                    Atrial fibrillation             0.4721
 Date of ablation     2     187            Atrial fibrillation episode             0.4715
 Date of ablation     3      28                                decimal             0.4523
Type of abalation     1     187            Atrial fibrillation episode             0.5795
Type of abalation     2       9                    Atrial fibrillation             0.5658
Type of abalation     3     172 Atrial fibrillation episode prediction             0.5435
              Age     1     335                              Age group             0.4673
          

In [8]:
print(len(sim_df))
print(sim_df.to_string())

# print(high_conf_df.to_string(index=False))

240
                                                                             metadata_col  rank afmo_id                                     afmo_label  cosine_similarity
0                                                                               Sample ID     1      76                                        decimal             0.4512
1                                                                               Sample ID     2     530                                        decimal             0.4512
2                                                                               Sample ID     3      28                                        decimal             0.4512
3                                                                        Date of ablation     1       9                            Atrial fibrillation             0.4721
4                                                                        Date of ablation     2     187                    Atrial fibrillation epi

In [9]:
# threshold = 0.45
threshold = 0.45
high_conf_df = sim_df[sim_df["cosine_similarity"] >= threshold]

high_conf_afmo_label = set()
for label in high_conf_df["afmo_label"]:
    high_conf_afmo_label.add(label)
print(len(high_conf_afmo_label))
print(high_conf_afmo_label)

50
{'Atrial fibrillation health risk assessment', 'ECG sensor', 'Pharmacotherapy', 'Sleep apnoea', 'ECG factor', 'Echocardiography', 'Health device', 'Body weight observation', 'Diabetes mellitus status', 'Electrocardiography', 'Thing', 'Medical condition', 'Atrial fibrillation episode prediction status', 'Clinical risk score', 'decimal', 'Stadiometer', 'BMI category', 'Hypertension', 'Race or ethnicity', 'Sleep apnoea status', 'Book', 'dateTimeStamp', 'Smoking status', 'Atrial fibrillation episode prediction', 'Data explanation', 'Excessive alcohol avoidance', 'Hypertension status', 'Diabetes mellitus', 'Sex', 'string', 'Body weight scale', 'Publication', 'Vital sign', 'Angiotensin receptor blocker therapy', 'Atrial fibrillation', 'Wearable device', 'duration', 'Atrial fibrillation episode', 'Chronic obstructive pulmonary disease status', 'Alcohol intake avoidance', 'ECG device', 'Prolonged PR interval', 'Age group', 'Chronic obstructive pulmonary disease', 'Alcohol misuse', 'Cardiova

In [ ]:
print("Total rows in sim_df:", len(sim_df))
print("Total rows in high_conf_df:", len(high_conf_df))
print("Unique AFMO labels in high_conf_afmo_label:", len(high_conf_afmo_label))

# which AFMO labels showed up most often across the 240 rows
print("\nMost frequent AFMO labels in top-3 matches:")
print(high_conf_df["afmo_label"].value_counts().head(len(high_conf_df)))

Total rows in sim_df: 240
Total rows in high_conf_df: 165
Unique AFMO labels in high_conf_afmo_label: 50

Most frequent AFMO labels in top-3 matches:
afmo_label
decimal                                          29
Atrial fibrillation episode                      15
Atrial fibrillation                              13
ECG device                                        9
Atrial fibrillation episode prediction            8
Medical condition                                 6
string                                            6
Angiotensin receptor blocker therapy              4
BMI category                                      4
Data explanation                                  4
Electrocardiography                               4
duration                                          3
Chronic obstructive pulmonary disease             3
Echocardiography                                  3
ECG factor                                        3
Body weight scale                                 3
Hyperte

Create onotology.json with the intersecting classes

In [ ]:
TEST_TARGET_LABELS = {
    "Age group",
    "Demographic factor",
    "Sex",
    "Race or ethnicity",
    "Atrial fibrillation",
}

HIGH_CONF_AFMO_LABELS = high_conf_afmo_label

TARGET_LABELS = {
    "Age group",
    "Demographic factor",
    "Sex",
    "Race or ethnicity",
    "Atrial fibrillation",
    "Atrial fibrillation episode",
    "Health event",
    "Diagnostic procedure",
    "Diagnostic procedure of cardiovascular system",
    "Electrocardiography",
    "ECG device",
    "Echocardiography",
    "Health wearable",
    "Wearable device",
    "Health sensor",
    "Sensor",
    "Cardiovascular disease",
    "Cardiovascular disease status",
    "Medical condition",
    "Hypertension",
    "Hypertension status",
    "Diabetes mellitus",
    "Diabetes mellitus status",
    "Chronic obstructive pulmonary disease",
    "Chronic obstructive pulmonary disease status",
    "Sleep apnoea",
    "Sleep apnoea status",
    "BMI category",
    "Anthropometric factor",
    "Anthropometric property",
    "Body height observation",
    "Body weight observation",
    "Health risk factor",
    "Clinical risk score",
    "Atrial fibrillation health risk assessment",
    "Atrial fibrillation risk status",
    "Health risk assessment",
    "Kidney disease",
    "Alcohol intake avoidance",
    "Alcohol misuse",
    "Alcoholism counselling",
    "Lifestyle factor",
    "Smoking status",
    "Pharmacotherapy",
    "Medical action",
    "Health device",
    "Continuous observation",
    "Discrete observation",
    "Observation",
    "Biometric property",
}
# print(len(TARGET_LABELS))

def get_display_label(attr):
    label = attr.get("label")
    if isinstance(label, str) and label.strip():
        return label.strip()
    if isinstance(label, dict):
        for key in ("en", "undefined", "IRI-based"):
            val = label.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        for val in label.values():
            if isinstance(val, str) and val.strip():
                return val.strip()
    iri = attr.get("iri", "")
    return iri.split("#")[-1] if "#" in iri else iri.split("/")[-1]

with open("docs/webvowl/data/ontology_full.json") as f:
    data = json.load(f)

# active_labels = {lbl.lower() for lbl in TEST_TARGET_LABELS}
active_labels = {lbl.lower() for lbl in HIGH_CONF_AFMO_LABELS}

kept_ids = set()
filtered_class_attrs = []

for attr in data.get("classAttribute", []):
    disp = get_display_label(attr)
    if disp.lower() in active_labels:
        kept_ids.add(attr["id"])
        filtered_class_attrs.append(attr)
        print(f"Matched id={attr['id']} -> {disp}")

filtered_classes = [c for c in data.get("class", []) if c.get("id") in kept_ids]

kept_prop_ids = set()
filtered_prop_attrs = []

for pattr in data.get("propertyAttribute", []):
    dom = pattr.get("domain")
    rng = pattr.get("range")
    if dom in kept_ids and rng in kept_ids:
        kept_prop_ids.add(pattr["id"])
        filtered_prop_attrs.append(pattr)

filtered_props = [p for p in data.get("property", []) if p.get("id") in kept_prop_ids]

data["class"] = filtered_classes
data["classAttribute"] = filtered_class_attrs
data["property"] = filtered_props
data["propertyAttribute"] = filtered_prop_attrs

with open("docs/webvowl/data/ontology.json", "w") as f:
    json.dump(data, f, indent=2)

print(f"Kept {len(kept_ids)} classes and {len(kept_prop_ids)} internal properties.")

Matched id=6 -> dateTimeStamp
Matched id=13 -> duration
Matched id=20 -> string
Matched id=28 -> decimal
Matched id=43 -> dateTimeStamp
Matched id=52 -> string
Matched id=60 -> string
Matched id=65 -> string
Matched id=73 -> dateTimeStamp
Matched id=76 -> decimal
Matched id=92 -> string
Matched id=99 -> duration
Matched id=104 -> dateTimeStamp
Matched id=96 -> decimal
Matched id=114 -> string
Matched id=128 -> Excessive alcohol avoidance
Matched id=130 -> string
Matched id=134 -> decimal
Matched id=135 -> string
Matched id=142 -> string
Matched id=151 -> dateTimeStamp
Matched id=157 -> decimal
Matched id=183 -> Diabetes mellitus
Matched id=206 -> duration
Matched id=209 -> decimal
Matched id=217 -> dateTimeStamp
Matched id=218 -> Stadiometer
Matched id=229 -> duration
Matched id=54 -> Body weight observation
Matched id=241 -> decimal
Matched id=268 -> Electrocardiography
Matched id=315 -> string
Matched id=317 -> string
Matched id=338 -> Sex
Matched id=129 -> Alcohol intake avoidance
M